# Klasik Görüntü İşleme ile Mikroskobik Görüntülerde Hücre Segmentasyonu, Sayımı ve Zamansal Analizi

**Görüntü İşleme — Yüksek Lisans Ara Sınav Projesi**
**Öğrenci:** Murat Sağlam — **No:** 264327008
**Seed:** `27008 + 2026 = 29034` (öğrenci numarasının son 5 hanesi + 2026)
**Veri seti:** Cell Tracking Challenge — Fluo-N2DH-GOWT1 (eğitim arşivi)

Bu notebook ödevdeki adımları sırasıyla uygular. Bütün işlem kodları `src/cellcount/` modüllerindedir;
notebook bu modülleri çağırır, tabloları ve şekilleri üretir.

> **Test güvenliği:** Test (Sequence 02) referans sayıları yalnızca **Bölüm 12**'deki kilitleme adımından
> sonra okunabilir. `references.load_test_references()` fonksiyonu kilit dosyalarının SHA-256 özetlerini
> doğrulamadan çalışmaz.

## 0. Kurulum

In [1]:
import sys, os, json, platform, urllib.request, zipfile
from pathlib import Path
REPO = Path.cwd()
sys.path.insert(0, str(REPO / "src"))
import pandas as pd, numpy as np, cv2, skimage, scipy, matplotlib
from IPython.display import Image, display
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 60)
from cellcount import config, data, selection, references, eda, experiments, validation, testing, figures, pipeline, posthoc
from cellcount import run_all as R
print("Python", platform.python_version(), "| OpenCV", cv2.__version__, "| scikit-image", skimage.__version__,
      "| SciPy", scipy.__version__, "| NumPy", np.__version__, "| pandas", pd.__version__)
print("Seed:", config.SEED, "| Veri klasörü:", config.DATA_DIR)

Python 3.14.0 | OpenCV 5.0.0 | scikit-image 0.26.0 | SciPy 1.18.1 | NumPy 2.5.4 | pandas 3.1.1
Seed: 29034 | Veri klasörü: C:\Murat\hucre-sayimi\data\Fluo-N2DH-GOWT1


In [2]:
# (Google Colab için) Veri seti yoksa resmi bağlantıdan indirilip data/ altına açılır.
DOWNLOAD_IF_MISSING = True
if not (config.DATA_DIR / "01").is_dir() and DOWNLOAD_IF_MISSING:
    config.DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    zpath = config.DATA_DIR.parent / "Fluo-N2DH-GOWT1.zip"
    print("İndiriliyor:", config.DATASET_URL)
    urllib.request.urlretrieve(config.DATASET_URL, zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(config.DATA_DIR.parent)
data.check_dataset(); print("Veri seti hazır.")

Veri seti hazır.


## 2. Veri setinin ön incelemesi
Sequence 01 ve 02'nin boyut, veri tipi, yoğunluk ve histogram özellikleri. Hücre boyut/parlaklık
istatistikleri yalnızca Sequence 01 (geliştirme + doğrulama) anotasyonlarından hesaplanır.

In [3]:
stats = R.stage_dataset()
stats

,sequence,n_frames,shape,dtype,min,max,mean,std,p1,p50,p99,p99_9,zero_fraction,p99_9_per_frame_min,p99_9_per_frame_max
0,01,92,1024x1024,uint8,0,255,3.97,12.57,0.0,0.0,68.0,139.0,0.573,125.0,157.0
1,02,92,1024x1024,uint8,0,182,2.09,5.61,0.0,0.0,31.0,52.0,0.618,49.0,57.0


In [4]:
display(Image(str(config.FIG_DIR / "fig_dataset_histograms.png"), width=700))

## 3. Geliştirme, doğrulama ve test görüntülerinin seçilmesi (seed = 29034)

In [5]:
sel = R.stage_selection()
print("Seçim kuralları doğrulandı (çakışma yok, gruplar/bloklar arası >= 5 kare boşluk).")
sel

Seçim kuralları doğrulandı (çakışma yok, gruplar/bloklar arası >= 5 kare boşluk).


,image_id,group,sequence,frame,test_block,attempts,file_name,tra_file,seed
0,DEV01,development,01,0,,1,01/t000.tif,01_GT/TRA/man_track000.tif,29034
1,DEV02,development,01,3,,1,01/t003.tif,01_GT/TRA/man_track003.tif,29034
2,DEV03,development,01,13,,1,01/t013.tif,01_GT/TRA/man_track013.tif,29034
3,DEV04,development,01,31,,1,01/t031.tif,01_GT/TRA/man_track031.tif,29034
4,DEV05,development,01,33,,1,01/t033.tif,01_GT/TRA/man_track033.tif,29034
5,DEV06,development,01,35,,1,01/t035.tif,01_GT/TRA/man_track035.tif,29034
6,DEV07,development,01,66,,1,01/t066.tif,01_GT/TRA/man_track066.tif,29034
7,DEV08,development,01,69,,1,01/t069.tif,01_GT/TRA/man_track069.tif,29034
8,DEV09,development,01,71,,1,01/t071.tif,01_GT/TRA/man_track071.tif,29034
9,DEV10,development,01,75,,1,01/t075.tif,01_GT/TRA/man_track075.tif,29034


## 4. Referans hücre sayıları (geliştirme + doğrulama)

In [6]:
refs, cells = R.stage_references(sel)
display(refs)
print(cells[["marker_area_px", "eq_diameter_px", "mean_intensity"]].describe().round(1))
print("Kenara değen referans hücre sayısı:", int(cells.touches_border.sum()))
print("Ortalama yoğunluğu < 10 olan (çok soluk) hücre oranı:", round(float((cells.mean_intensity < 10).mean()), 3))

,image_id,group,sequence,frame,ref_count,min_marker_area,median_marker_area,n_small_markers_lt300px
0,DEV01,development,01,0,23,1125,1789.0,0
1,DEV02,development,01,3,23,1181,1694.0,0
2,DEV03,development,01,13,23,223,1649.0,1
3,DEV04,development,01,31,24,972,1650.5,0
4,DEV05,development,01,33,24,971,1712.0,0
5,DEV06,development,01,35,24,696,1649.0,0
6,DEV07,development,01,66,21,889,1649.0,0
7,DEV08,development,01,69,21,1115,1649.0,0
8,DEV09,development,01,71,21,889,1649.0,0
9,DEV10,development,01,75,21,1162,1694.0,0


       marker_area_px  eq_diameter_px  mean_intensity
count           332.0           332.0           332.0
mean           1822.0            47.4            36.0
std             663.7             8.5            25.7
min              45.0             7.6             3.2
25%            1418.2            42.5            16.9
50%            1649.0            45.8            30.3
75%            1916.0            49.4            47.1
max            4522.0            75.9           111.8
Kenara değen referans hücre sayısı: 0
Ortalama yoğunluğu < 10 olan (çok soluk) hücre oranı: 0.123


## 5–10. Baseline ve kontrollü deneyler (yalnızca 10 geliştirme görüntüsü)
* **Bölüm 5 — Baseline:** gri seviye → Otsu → bağlı bileşen analizi → sayım. Minimum alan eşiği geliştirme MAE'sine göre seçilir.
* **Bölüm 6 — Yoğunluk dönüşümleri**, **Bölüm 7 — Filtreler**, **Bölüm 8 — Segmentasyon**, **Bölüm 9 — Morfoloji / ayırma**:
  her grupta sabit bir referans ayarı korunur ve yalnızca incelenen faktör değiştirilir.

In [7]:
exp, exp_pf, baseline_min_area, calib = R.stage_experiments(sel)
print("Baseline için seçilen minimum alan:", baseline_min_area)
calib

Baseline için seçilen minimum alan: 50


,min_area,MAE,RMSE,Bias
0,0,847.2,863.022016,847.2
1,10,37.7,39.088361,37.7
2,20,12.1,13.524053,10.1
3,30,7.4,7.861298,1.4
4,50,5.2,7.293833,-4.6
5,75,7.2,8.648699,-7.2
6,100,8.2,9.402127,-8.2
7,150,9.5,10.397115,-9.5
8,300,10.9,11.388591,-10.9


In [8]:
cols = ["experiment_id", "group", "changed_factor", "intensity", "filter", "segmentation", "morphology",
        "params", "count_MAE", "count_RMSE", "bias", "exact_acc", "mean_time_s", "missed", "false_pos",
        "merged_objs", "split_cells"]
exp[cols]

,experiment_id,group,changed_factor,intensity,filter,segmentation,morphology,params,count_MAE,count_RMSE,bias,exact_acc,mean_time_s,missed,false_pos,merged_objs,split_cells
0,E01,5-Baseline,"Baseline (ham, Otsu, CCL)",none,none,otsu,yok,min_area=50,5.2,7.294,-4.6,0.1,0.0332,91,1,0,22
1,E02,6-Yoğunluk,Dönüşüm yok (orijinal),none,gaussian(ksize=9),otsu,open7 -> close11 -> fill_holes,min_area=300; kenar nesneleri çıkarıldı; filtre->yoğunluk,10.8,11.269,-10.8,0.0,0.0534,108,0,0,0
2,E03,6-Yoğunluk,Contrast stretching,"stretch(p_low=1.0, p_high=99.8)",gaussian(ksize=9),otsu,open7 -> close11 -> fill_holes,min_area=300; kenar nesneleri çıkarıldı; filtre->yoğunluk,10.5,11.041,-10.5,0.0,0.0625,105,0,0,0
3,E04,6-Yoğunluk,Histogram equalization,histeq,gaussian(ksize=9),otsu,open7 -> close11 -> fill_holes,min_area=300; kenar nesneleri çıkarıldı; filtre->yoğunluk,6.6,7.681,-0.8,0.0,0.0519,119,133,16,0
4,E05,6-Yoğunluk,CLAHE,"clahe(clip_limit=2.0, tile=8)",gaussian(ksize=9),otsu,open7 -> close11 -> fill_holes,min_area=300; kenar nesneleri çıkarıldı; filtre->yoğunluk,6.3,6.535,-6.3,0.0,0.0536,63,0,0,0
5,E06,6-Yoğunluk,Histogram matching,match,gaussian(ksize=9),otsu,open7 -> close11 -> fill_holes,min_area=300; kenar nesneleri çıkarıldı; filtre->yoğunluk,12.1,12.211,-12.1,0.0,0.0622,121,0,0,0
6,E07,6-Yoğunluk,Gamma (0.5),"gamma(gamma=0.5, p_high=99.9)",gaussian(ksize=9),otsu,open7 -> close11 -> fill_holes,min_area=300; kenar nesneleri çıkarıldı; filtre->yoğunluk,6.3,6.535,-6.3,0.0,0.0635,63,0,0,0
7,E08,6-Yoğunluk,Log dönüşümü,log(p_high=99.9),gaussian(ksize=9),otsu,open7 -> close11 -> fill_holes,min_area=300; kenar nesneleri çıkarıldı; filtre->yoğunluk,3.6,3.924,-3.6,0.0,0.0631,31,0,5,0
8,E09,7-Filtre,Filtre yok,log(p_high=99.9),none,otsu,open7 -> close11 -> fill_holes,min_area=300; kenar nesneleri çıkarıldı; filtre->yoğunluk,3.4,3.688,-3.4,0.0,0.0658,30,0,4,0
9,E10,7-Filtre,Gaussian 3x3,log(p_high=99.9),gaussian(ksize=3),otsu,open7 -> close11 -> fill_holes,min_area=300; kenar nesneleri çıkarıldı; filtre->yoğunluk,3.9,4.254,-3.9,0.0,0.0649,36,0,3,0


In [9]:
for g, d in exp.groupby("group", sort=False):
    best = d.sort_values(["count_MAE", "count_RMSE"]).iloc[0]
    print(f"{g:16s} en iyi: {best.experiment_id} {best.changed_factor:40s} MAE={best.count_MAE:.2f}")

5-Baseline       en iyi: E01 Baseline (ham, Otsu, CCL)                MAE=5.20
6-Yoğunluk       en iyi: E08 Log dönüşümü                             MAE=3.60
7-Filtre         en iyi: E12 Gaussian 13x13                           MAE=3.40
8-Segmentasyon   en iyi: E25 Adaptif Gauss (151, C=-20)               MAE=0.70
9-Morfoloji      en iyi: E38 Opening 7 -> Closing 11                  MAE=0.70


### Deney grafikleri

In [10]:
config.ensure_dirs()
ctx = experiments.build_context(sel)
demo = sel[sel.group == "development"].iloc[3]
demo_ref = int(refs.set_index("image_id").loc[demo.image_id, "ref_count"])
figures.fig_experiments_mae(exp); figures.fig_param_sensitivity(exp, calib)
for p in ["fig_experiments_mae.png", "fig_parameter_sensitivity.png"]:
    display(Image(str(config.FIG_DIR / p), width=1000))

## 11. Doğrulama ve nihai pipeline seçimi

In [11]:
for k, v in experiments.candidate_pipelines().items():
    print(f"{k}: {v['desc']}")
val_df, winner, final_doc = R.stage_validation(sel, baseline_min_area)
val_df

Pipeline A: Global eşik: G9 -> log -> Otsu -> open7/close11/fill
Pipeline B: Yerel eşik: G9 -> log -> adaptif Gauss -> open7/close11/fill
Pipeline C: Pipeline B + distance transform/h-maxima watershed
Pipeline D: Arka plan çıkarma: G9 -> log -> top-hat(81) -> Otsu -> open7/close11/fill


,pipeline,description,val_MAE,val_RMSE,val_bias,val_exact_acc,mean_time_s,complexity_steps,missed,false_pos,merged_objs,split_cells,selected,rank
0,Pipeline A,Global eşik: G9 -> log -> Otsu -> open7/close11/fill,3.0,3.376,-3.0,0.0,0.0641,7,14,0,1,0,,3.0
1,Pipeline B,Yerel eşik: G9 -> log -> adaptif Gauss -> open7/close11/...,0.8,0.894,-0.8,0.2,0.1044,7,6,2,0,0,SEÇİLDİ,1.0
2,Pipeline C,Pipeline B + distance transform/h-maxima watershed,0.8,0.894,-0.8,0.2,0.2989,8,6,2,0,0,,2.0
3,Pipeline D,Arka plan çıkarma: G9 -> log -> top-hat(81) -> Otsu -> o...,3.0,3.376,-3.0,0.0,0.1332,8,15,1,1,0,,4.0
4,Baseline,Referans amaçlı (seçime katılmaz),2.8,3.521,1.2,0.2,0.0332,1,33,0,0,18,(referans),NaN


In [12]:
print("Seçilen nihai pipeline:", winner)
print(json.dumps(final_doc["final_pipeline"], indent=2, ensure_ascii=False))
figures.fig_validation(val_df); figures.fig1_flow(final_doc)
display(Image(str(config.FIG_DIR / "fig_validation.png"), width=650))
display(Image(str(config.FIG_DIR / "fig1_final_pipeline_flow.png"), width=1000))

Seçilen nihai pipeline: Pipeline B
{
  "intensity": {
    "method": "log",
    "p_high": 99.9
  },
  "filter": {
    "method": "gaussian",
    "ksize": 9
  },
  "order": "filter_first",
  "segmentation": {
    "method": "adaptive_gaussian",
    "block_size": 151,
    "C": -20
  },
  "morphology": [
    {
      "op": "open",
      "size": 7
    },
    {
      "op": "close",
      "size": 11
    },
    {
      "op": "fill_holes"
    }
  ],
  "separation": {
    "method": "none"
  },
  "min_area": 300,
  "clear_border": true,
  "name": "Pipeline B"
}


In [13]:
# Görüntü içeren şekiller (rapor için; GitHub sürümünde çıktıları temizlenir)
paths = [figures.fig2_intensity(int(demo.frame), ctx), figures.fig_filters(int(demo.frame)),
         figures.fig3_seg_morph(int(demo.frame), demo_ref, ctx), figures.fig4_steps(final_doc, int(demo.frame), ctx),
         figures.fig_dataset_samples(int(demo.frame), int(sel[sel.group == "test"].frame.iloc[0]))]
for p in paths:
    display(Image(str(p), width=1000))

## 12. Nihai pipeline'ın kilitlenmesi ve test tahminlerinin kaydedilmesi
Bu hücre `final_pipeline.json` dosyasını okur ve 15 test görüntüsünde baseline ile nihai pipeline'ı
**test referanslarını okumadan** çalıştırır. Ardından `test_predictions_locked.csv` ve SHA-256 özetlerini
içeren `test_lock_manifest.json` dosyalarını yazar.

> **Bu noktada** `final_pipeline.json`, `test_predictions_locked.csv`, `test_lock_manifest.json` ve kaynak
> kodları **ayrı bir commit** ile GitHub'a gönderilir (bkz. `GIT_ADIMLARI.md`).

In [14]:
locked = R.stage_lock(sel)
print(json.dumps(testing.verify_lock(), indent=2, ensure_ascii=False))
locked

{
  "locked_at_utc": "2026-11-14T14:24:15+00:00",
  "final_pipeline_sha256": "6890be1150a4c7f5131ae23b053f84d50353ec2e608eb136ad2cb9cdeec76b44",
  "test_predictions_sha256": "9a42da0a9d970730587c5d9b529f4faf83b019d98c2ad63609a92beeed028868",
  "selected_frames_sha256": "709bfc74d969281da8dc89237ce2d230790ca61900c20fcabe9fddeaefcb94c3",
  "note": "Test referansları bu dosya oluşturulmadan önce okunmamıştır."
}


,image_id,sequence,frame,test_block,baseline_pred,final_pred,baseline_time_s,final_time_s
0,TST01,2,22,B1,20,36,0.0329,0.1030
1,TST02,2,23,B1,23,38,0.0329,0.1037
2,TST03,2,24,B1,20,39,0.0330,0.1033
3,TST04,2,25,B1,18,39,0.0330,0.1032
4,TST05,2,26,B1,18,37,0.0333,0.1027
5,TST06,2,35,B2,21,40,0.0359,0.1022
6,TST07,2,36,B2,20,38,0.0330,0.1034
7,TST08,2,37,B2,23,42,0.0333,0.1045
8,TST09,2,38,B2,23,43,0.0334,0.1023
9,TST10,2,39,B2,22,40,0.0332,0.1035


## 13. Bağımsız test performansı (kilit sonrası)

In [15]:
test_df, summary, temporal, obj_err = R.stage_evaluate(sel)
test_df[["frame_label", "image_id", "frame", "test_block", "ref_count", "baseline_pred", "final_pred",
         "baseline_abs_err", "final_abs_err"]]

,frame_label,image_id,frame,test_block,ref_count,baseline_pred,final_pred,baseline_abs_err,final_abs_err
0,Frame 1,TST01,22,B1,24,20,36,4,12
1,Frame 2,TST02,23,B1,24,23,38,1,14
2,Frame 3,TST03,24,B1,23,20,39,3,16
3,Frame 4,TST04,25,B1,25,18,39,7,14
4,Frame 5,TST05,26,B1,25,18,37,7,12
5,Frame 6,TST06,35,B2,26,21,40,5,14
6,Frame 7,TST07,36,B2,27,20,38,7,11
7,Frame 8,TST08,37,B2,27,23,42,4,15
8,Frame 9,TST09,38,B2,28,23,43,5,15
9,Frame 10,TST10,39,B2,28,22,40,6,12


In [16]:
summary

,method,MAE,RMSE,Bias,ExactCountAccuracy,MeanTime_s
0,Baseline,5.667,5.983,-5.667,0.0,0.0333
1,Nihai pipeline,12.667,12.967,12.667,0.0,0.1032


In [17]:
obj_err.groupby("method")[["missed", "false_pos", "merged", "merged_cells", "split"]].sum()

,missed,false_pos,merged,merged_cells,split
method,,,,,
Baseline,183,7,0,0,42
Nihai pipeline,16,211,5,10,0


## 14. Ardışık görüntülerde zamansal analiz

In [18]:
figures.fig5_temporal(test_df)
display(Image(str(config.FIG_DIR / "fig5_temporal_counts.png"), width=1000))
temporal

,test_block,frames,series,counts,deltas,mean_abs_delta,n_changes,range
0,B1,22-26,Gerçek,24 24 23 25 25,+0 -1 +2 +0,0.75,2,2.0
1,B1,22-26,Baseline,20 23 20 18 18,+3 -3 -2 +0,2.00,3,5.0
2,B1,22-26,Nihai pipeline,36 38 39 39 37,+2 +1 +0 -2,1.25,3,3.0
3,B1,22-26,Baseline Δ-uyumu,,,2.25,2,NaN
4,B1,22-26,Nihai pipeline Δ-uyumu,,,2.00,4,NaN
5,B2,35-39,Gerçek,26 27 27 28 28,+1 +0 +1 +0,0.50,2,2.0
6,B2,35-39,Baseline,21 20 23 23 22,-1 +3 +0 -1,1.25,3,3.0
7,B2,35-39,Nihai pipeline,40 38 42 43 40,-2 +4 +1 -3,2.50,4,5.0
8,B2,35-39,Baseline Δ-uyumu,,,1.75,4,NaN
9,B2,35-39,Nihai pipeline Δ-uyumu,,,2.50,3,NaN


## 15. Başarılı / hatalı / zorlayıcı örnekler

In [19]:
p = figures.fig6_examples(final_doc, test_df, ctx)
display(Image(str(p), width=1000))

### Kilit sonrası hata analizi (yalnızca yorum amaçlı)
Kilitli tahminler **değiştirilmez**. Aşağıdaki analiz, nihai pipeline'ın testte neden fazla saydığını
açıklamak için Sequence 01 ve 02'nin log-uzayı arka plan gürültüsünü ve yanlış pozitif nesnelerin şeklini
karşılaştırır.

In [20]:
display(pd.read_csv(config.REPO_ROOT / "posthoc_background_noise.csv"))
display(pd.read_csv(config.REPO_ROOT / "posthoc_object_shapes.csv"))
# Seçilmeyen adayların testte durumu (yalnızca bilgi; seçim/kilit değişmez)
display(pd.read_csv(config.REPO_ROOT / "posthoc_candidates_on_test.csv"))

,group,sequence,raw_p99_9,bg_mean_log,bg_std_log,cell_median_log,cell_bg_contrast_in_std
0,development,1,136.3,47.51,14.84,186.2,9.35
1,test,2,53.2,40.53,21.68,170.0,5.97


,group,kind,n,area_median,solidity_median,solidity_p10,solidity_p90
0,development,FP,1,507.0,0.596,0.596,0.596
1,development,TP,217,3066.0,0.980,0.941,0.985
2,test,FP,211,481.0,0.660,0.470,0.880
3,test,TP,381,3010.0,0.977,0.905,0.984


,pipeline,val_MAE,test_MAE_posthoc,test_RMSE_posthoc,test_bias_posthoc,selected
0,Pipeline A,3.0,2.667,3.033,-2.533,NaN
1,Pipeline B,0.8,12.667,12.967,12.667,SEÇİLDİ (kilitli)
2,Pipeline C,0.8,17.000,17.445,17.000,NaN
3,Pipeline D,3.0,3.133,3.568,-3.133,NaN


In [21]:
# Doğrulama ve test karşılaştırması
v = val_df.set_index("pipeline")
s = summary.set_index("method")
pd.DataFrame({"Doğrulama MAE": [v.loc["Baseline", "val_MAE"], v.loc[winner, "val_MAE"]],
              "Test MAE": [s.loc["Baseline", "MAE"], s.loc["Nihai pipeline", "MAE"]],
              "Doğrulama RMSE": [v.loc["Baseline", "val_RMSE"], v.loc[winner, "val_RMSE"]],
              "Test RMSE": [s.loc["Baseline", "RMSE"], s.loc["Nihai pipeline", "RMSE"]]},
             index=["Baseline", f"Nihai ({winner})"])

,Doğrulama MAE,Test MAE,Doğrulama RMSE,Test RMSE
Baseline,2.8,5.667,3.521,5.983
Nihai (Pipeline B),0.8,12.667,0.894,12.967


Bütün sayısal sonuçlar repo kökündeki CSV dosyalarına yazılmıştır: `selected_frames.csv`, `experiments.csv`,
`validation_results.csv`, `final_pipeline.json`, `test_predictions_locked.csv`, `test_results.csv`,
`test_summary.csv`, `temporal_analysis.csv`. Yorum ve tartışma teknik raporda yer almaktadır.